In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses 
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from torch.utils.data import DataLoader
from datasets import Dataset,load_dataset
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.losses import MultipleNegativesRankingLoss


/home/harikrishnan/venvs/nco_sem_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
full_dataset = load_dataset("csv", data_files='../data/processed/nco_cleaned.csv',split="train")


Generating train split: 3375 examples [00:00, 27938.19 examples/s]


In [8]:
def filter_function(example):
    return example["Unit_Title"] is not None and example["Unit_Description"] is not None
filtered_dataset = full_dataset.filter(filter_function)
train_dataset = filtered_dataset.select_columns(["Unit_Description", "Unit_Title"])
print("Dataset is ready for training:")
print(train_dataset)

Filter: 100%|██████████| 3375/3375 [00:00<00:00, 54559.03 examples/s]

Dataset is ready for training:
Dataset({
    features: ['Unit_Description', 'Unit_Title'],
    num_rows: 3375
})


In [9]:
dataset_dict = train_dataset.train_test_split(test_size=0.1, seed=42)

# The result is a dictionary containing the two splits
train_split = dataset_dict['train']
eval_split = dataset_dict['test']
print(train_split)

Dataset({
    features: ['Unit_Description', 'Unit_Title'],
    num_rows: 3037
})


In [14]:
model = SentenceTransformer("taronaeo/all-MiniLM-L6-v2-BE")

# Initialize the evaluator


2025-09-13 17:00:59 - Use pytorch device_name: cuda:0
2025-09-13 17:00:59 - Load pretrained SentenceTransformer: taronaeo/all-MiniLM-L6-v2-BE
2025-09-13 17:01:00 - No sentence-transformers model found with name taronaeo/all-MiniLM-L6-v2-BE. Creating a new one with mean pooling.


ValueError: Unrecognized model in taronaeo/all-MiniLM-L6-v2-BE. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, altclip, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, colpali, conditional_detr, convbert, convnext, convnextv2, cpmant, csm, ctrl, cvt, d_fine, dab-detr, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deepseek_v3, deformable_detr, deit, depth_anything, depth_pro, deta, detr, diffllama, dinat, dinov2, dinov2_with_registers, distilbert, donut-swin, dpr, dpt, efficientformer, effi

In [10]:
loss = MultipleNegativesRankingLoss(model)
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="models/nco_bge_fintunes",
    # Optional training parameters:
    num_train_epochs=6,
    do_eval=True
      # Used in W&B if `wandb` is installed
)

In [11]:
trainer = SentenceTransformerTrainer(
    model=model,
    train_dataset=train_split,
    loss=loss,args=args,
    eval_dataset=eval_split
    
)

In [12]:
trainer.train()


Step,Training Loss
500,0.004000
1000,0.001100
1500,0.000800
2000,0.000400


TrainOutput(global_step=2280, training_loss=0.0014639216426171754, metrics={'train_runtime': 435.6517, 'train_samples_per_second': 41.827, 'train_steps_per_second': 5.234, 'total_flos': 0.0, 'train_loss': 0.0014639216426171754, 'epoch': 6.0})

In [12]:
import logging
import time
import numpy as np
from datasets import load_dataset

from sentence_transformers import (
    SentenceTransformer,
    export_dynamic_quantized_onnx_model,
    export_static_quantized_openvino_model,
)
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

#### Just some code to print debug information to stdout
logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)

# Load some sentences from the STSbenchmark dataset

model_name = "/home/harikrishnan/Coding/Statathon/nco-semantic-search/models/finetunedenglishmodel"

# 1. Load a baseline model with just fp32 torch
model = SentenceTransformer(model_name, device="cpu")

# 2. Load an ONNX model to quantize
onnx_model = SentenceTransformer(
    model_name,
    backend="onnx",
    device="cpu",
    model_kwargs={"provider": "CPUExecutionProvider"},
)

# 3. Quantize the ONNX model
quantized_onnx_model_path = f"{model_name.replace('/', '-')}-onnx-quantized"
onnx_model.save_pretrained(quantized_onnx_model_path)
export_dynamic_quantized_onnx_model(
    onnx_model,
    quantization_config="avx512_vnni",
    model_name_or_path=quantized_onnx_model_path,
)
quantized_onnx_model = SentenceTransformer(
    quantized_onnx_model_path,
    backend="onnx",
    device="cpu",
    model_kwargs={
        "file_name": "model_qint8_avx512_vnni.onnx",
        "provider": "CPUExecutionProvider",
    },
)
# Alternatively, you can load the pre-quantized model:
# quantized_onnx_model = SentenceTransformer(
#     model_name,
#     backend="onnx",
#     device="cpu",
#     model_kwargs={
#         "file_name": "model_qint8_avx512_vnni.onnx",
#         "provider": "CPUExecutionProvider",
#     },
# )

# To make sure that `onnx_model` itself didn't get quantized, we reload it
onnx_model = SentenceTransformer(
    model_name,
    backend="onnx",
    device="cpu",
    model_kwargs={"provider": "CPUExecutionProvider"},
)

# 4. Load an OpenVINO model to quantize
openvino_model = SentenceTransformer(model_name, backend="openvino", device="cpu")

# 5. Quantize the OpenVINO model
quantized_ov_model_path = f"{model_name.replace('/', '-')}-ov-quantized"
openvino_model.save_pretrained(quantized_ov_model_path)
export_static_quantized_openvino_model(
    openvino_model,
    quantization_config=None,
    model_name_or_path=quantized_ov_model_path,
)
quantized_ov_model = SentenceTransformer(
    quantized_ov_model_path,
    backend="openvino",
    device="cpu",
    model_kwargs={"file_name": "openvino_model_qint8_quantized.xml"},
)
# Alternatively, you can load the pre-quantized model:
# quantized_ov_model = SentenceTransformer(
#     model_name,
#     backend="openvino",
#     device="cpu",
#     model_kwargs={"file_name": "openvino_model_qint8_quantized.xml"},
# )

# To make sure that `openvino_model` itself didn't get quantized, we reload it
openvino_model = SentenceTransformer(model_name, backend="openvino", device="cpu")

# -------------------------------
def benchmark(model, sentences, n_runs=10, warmup=2):
    # Warmup runs
    for _ in range(warmup):
        _ = model.encode(sentences, convert_to_tensor=False)

    times = []
    for _ in range(n_runs):
        start = time.time()
        _ = model.encode(sentences, convert_to_tensor=False)
        end = time.time()
        times.append(end - start)

    avg = np.mean(times)
    std = np.std(times)
    return avg, std

# Test sentences
sentences = [
    "The cat sits outside",
    "A man is playing guitar",
    "The new movie is awesome",
    "I love machine learning",
]

# -------------------------------
# Run benchmarks
# -------------------------------
models_to_test = {
    "Torch FP32": model,
    "ONNX FP32": onnx_model,
    "ONNX Quantized": quantized_onnx_model,
    "OpenVINO FP32": openvino_model,
    "OpenVINO Quantized": quantized_ov_model,
}

results = {}
for name, mdl in models_to_test.items():
    logging.info(f"Benchmarking {name} ...")
    avg, std = benchmark(mdl, sentences)
    results[name] = (avg, std)
    logging.info(f"{name}: {avg*1000:.2f} ± {std*1000:.2f} ms per encode")

print("\n=== Latency Results ===")
for name, (avg, std) in results.items():
    print(f"{name:20s}: {avg*1000:.2f} ± {std*1000:.2f} ms")

2025-09-13 16:54:07 - Load pretrained SentenceTransformer: /home/harikrishnan/Coding/Statathon/nco-semantic-search/models/finetunedenglishmodel
2025-09-13 16:54:07 - Load pretrained SentenceTransformer: /home/harikrishnan/Coding/Statathon/nco-semantic-search/models/finetunedenglishmodel
2025-09-13 16:54:07 - No 'model.onnx' found in '/home/harikrishnan/Coding/Statathon/nco-semantic-search/models/finetunedenglishmodel'. Exporting the model to ONNX.
2025-09-13 16:54:08 - Saving the exported ONNX model is heavily recommended to avoid having to export it again. Do so with `model.save_pretrained('/home/harikrishnan/Coding/Statathon/nco-semantic-search/models/finetunedenglishmodel')`.
2025-09-13 16:54:08 - Save model to -home-harikrishnan-Coding-Statathon-nco-semantic-search-models-finetunedenglishmodel-onnx-quantized
2025-09-13 16:54:09 - Quantization parameters for tensor:"/embeddings/LayerNorm/Add_1_output_0" not specified
2025-09-13 16:54:09 - Quantization parameters for tensor:"/encoder

/home/harikrishnan/venvs/nco_sem_env/lib/python3.12/site-packages/rich/live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/harikrishnan/venvs/nco_sem_env/lib/python3.12/site-packages/rich/live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
2025-09-13 16:55:08 - Load pretrained SentenceTransformer: -home-harikrishnan-Coding-Statathon-nco-semantic-search-models-finetunedenglishmodel-ov-quantized
2025-09-13 16:55:08 - Load pretrained SentenceTransformer: /home/harikrishnan/Coding/Statathon/nco-semantic-search/models/finetunedenglishmodel
2025-09-13 16:55:08 - No 'openvino_model.xml' found in '/home/harikrishnan/Coding/Statathon/nco-semantic-search/models/finetunedenglishmodel'. Exporting the model to OpenVINO.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid usi


=== Latency Results ===
Torch FP32          : 11.11 ± 1.58 ms
ONNX FP32           : 7.40 ± 0.66 ms
ONNX Quantized      : 6.72 ± 2.22 ms
OpenVINO FP32       : 6.75 ± 0.59 ms
OpenVINO Quantized  : 4.70 ± 0.48 ms


In [13]:
results = {}
for name, mdl in models_to_test.items():
    logging.info(f"Benchmarking {name} ...")
    avg, std = benchmark(mdl, sentences)
    results[name] = (avg, std)
    logging.info(f"{name}: {avg*1000:.2f} ± {std*1000:.2f} ms per encode")

print("\n=== Latency Results ===")
for name, (avg, std) in results.items():
    print(f"{name:20s}: {avg*1000:.2f} ± {std*1000:.2f} ms")

2025-09-13 16:55:11 - Benchmarking Torch FP32 ...
Batches: 100%|██████████| 1/1 [00:00<00:00, 97.80it/s]
2025-09-13 16:55:12 - Torch FP32: 13.13 ± 4.36 ms per encode
2025-09-13 16:55:12 - Benchmarking ONNX FP32 ...
Batches: 100%|██████████| 1/1 [00:00<00:00, 211.79it/s]
2025-09-13 16:55:12 - ONNX FP32: 6.40 ± 0.39 ms per encode
2025-09-13 16:55:12 - Benchmarking ONNX Quantized ...
Batches: 100%|██████████| 1/1 [00:00<00:00, 268.33it/s]
2025-09-13 16:55:12 - ONNX Quantized: 6.18 ± 1.39 ms per encode
2025-09-13 16:55:12 - Benchmarking OpenVINO FP32 ...
Batches: 100%|██████████| 1/1 [00:00<00:00, 236.49it/s]
2025-09-13 16:55:12 - OpenVINO FP32: 5.58 ± 0.57 ms per encode
2025-09-13 16:55:12 - Benchmarking OpenVINO Quantized ...
Batches: 100%|██████████| 1/1 [00:00<00:00, 392.65it/s]
2025-09-13 16:55:12 - OpenVINO Quantized: 4.30 ± 0.39 ms per encode



=== Latency Results ===
Torch FP32          : 13.13 ± 4.36 ms
ONNX FP32           : 6.40 ± 0.39 ms
ONNX Quantized      : 6.18 ± 1.39 ms
OpenVINO FP32       : 5.58 ± 0.57 ms
OpenVINO Quantized  : 4.30 ± 0.39 ms


In [3]:
import pandas as pd

# Load your dataset
df = pd.read_csv("../data/processed/nco_cleaned.csv")
df['Unit_Description'] = df['Unit_Description'].fillna("")

# Corpus texts
corpus_texts = df['Unit_Description'].tolist()
corpus_titles = df['Unit_Title'].tolist()


In [4]:
import torch
# model2 = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True)
# corpus_embeddings2 =model2.encode(corpus_texts, convert_to_tensor=True)

In [5]:
queries = [
    "Nuclear Physicist",
    "Scientist who studies nuclear physics and atomic particles",
    "Scientist analyzing nuclear reactions for medical and energy applications",
    "Person performing experiments with isotopes and X-ray crystallography",
    "Scientist with atom work",
    "Researcher in atomic physics",
    "Scientist for nuclear experiments"
]

for query in queries:
    query_embedding = model.encode(query, convert_to_tensor=True)
    cos_scores = torch.nn.functional.cosine_similarity(query_embedding, corpus_embeddings)
    top_idx = torch.argmax(cos_scores).item()  # <-- convert tensor to int
    print("bge model finetuned ----")
    print("\nQuery:", query)
    print("Predicted Unit_Title:", corpus_titles[top_idx])
    print("Unit_Code:", df.iloc[top_idx]['Unit_Code'])
    print("Similarity score:", cos_scores[top_idx].item())
    print("Unit_Description (first 300 chars):", df.iloc[top_idx]['Unit_Description'][:300], "...")
    print("-----")
    # print("original ")
    # query_embedding = model2.encode(query, convert_to_tensor=True)
    # cos_scores = torch.nn.functional.cosine_similarity(query_embedding, corpus_embeddings2)
    # top_idx = torch.argmax(cos_scores).item()  # <-- convert tensor to int
    # print("\nQuery:", query)
    # print("Predicted Unit_Title:", corpus_titles[top_idx])
    # print("Unit_Code:", df.iloc[top_idx]['Unit_Code'])
    # print("Similarity score:", cos_scores[top_idx].item())
    # print("Unit_Description (first 300 chars):", df.iloc[top_idx]['Unit_Description'][:300], "...")
    # print("-----")



bge model finetuned ----

Query: Nuclear Physicist
Predicted Unit_Title: Physicist, Nuclear
Unit_Code: 2111.08
Similarity score: 0.7614233493804932
Unit_Description (first 300 chars): Physicist, Nuclear; Physicist, Atomic conducts theoretical and experimental studies and research in fields of nuclear physics to formulate and apply theories in atomic fields for peaceful purposes. Performs basic tasks similar to Physicist General, conducts research and studies in atomic fields such ...
-----
bge model finetuned ----

Query: Scientist who studies nuclear physics and atomic particles
Predicted Unit_Title: Physicist, Nuclear
Unit_Code: 2111.08
Similarity score: 0.7757603526115417
Unit_Description (first 300 chars): Physicist, Nuclear; Physicist, Atomic conducts theoretical and experimental studies and research in fields of nuclear physics to formulate and apply theories in atomic fields for peaceful purposes. Performs basic tasks similar to Physicist General, conducts research and studies i

In [2]:
import torch
from sentence_transformers import SentenceTransformer
import os

def quantize_sentence_transformer(model_path: str, output_path: str):
    """
    Loads a SentenceTransformer model, applies dynamic quantization, and saves it.

    Args:
        model_path (str): The path to the original fine-tuned model.
        output_path (str): The directory where the quantized model will be saved.
    """
    print(f"Loading model from: {model_path}")
    
    # 1. Load your fine-tuned sentence transformer model
    model = SentenceTransformer(model_path)

    print("Model loaded successfully. Starting quantization...")

    # 2. Access the underlying Transformer model
    transformer_module = model[0].auto_model

    # 3. Apply dynamic quantization
    quantized_transformer = torch.quantization.quantize_dynamic(
        model=transformer_module,
        qconfig_spec={torch.nn.Linear},
        dtype=torch.qint8
    )

    # 4. Replace the original transformer layer with the new quantized layer
    model[0].auto_model = quantized_transformer

    # 5. Save the quantized model
    print(f"Quantization complete. Saving model to: {output_path}")
    os.makedirs(output_path, exist_ok=True)
    
    # --- FIX IS HERE ---
    # Disable safetensors and use the standard PyTorch saving method
    model.save(output_path, safe_serialization=False)
    
    print(f"Successfully saved quantized model at {output_path}")


if __name__ == '__main__':
    # The path to your locally saved, fine-tuned model
    original_model_path = '/home/harikrishnan/Coding/Statathon/nco-semantic-search/models/finetunedenglishmodel'
    
    # The path where you want to save the new quantized model
    quantized_model_directory = './quantized_finetuned_model'
    
    quantize_sentence_transformer(original_model_path, quantized_model_directory)

Loading model from: /home/harikrishnan/Coding/Statathon/nco-semantic-search/models/finetunedenglishmodel
Model loaded successfully. Starting quantization...
Quantization complete. Saving model to: ./quantized_finetuned_model


AttributeError: 'torch.dtype' object has no attribute 'device'